# ENOE CDMX 2020-2025 — Análisis Exploratorio
## Desempleo y condiciones laborales en jóvenes de la Ciudad de México

In [0]:
%pip install klib
%matplotlib inline

In [0]:
dbutils.library.restartPython()

In [0]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import klib

matplotlib.use("Agg")
plt.rcParams["figure.figsize"] = (16, 8)
plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid")

TABLA = "workspace.tesis.enoe_jovenes_cdmx"
df = spark.table(TABLA).toPandas()

print(f"Filas    : {df.shape[0]:,}")
print(f"Columnas : {df.shape[1]}")
df.head()

In [0]:
COLS_NUMERICAS = [c for c in [
    "eda", "anios_esc", "n_hij",
    "clase1", "clase2", "clase3",
    "pos_ocu", "rama_est1", "seg_soc",
    "hrsocup", "ingocup", "ing_x_hrs",
    "dur_des", "sub_o", "desempleado", "pea",
    "periodo_ord",
] if c in df.columns]

COLS_CATEGORICAS = [c for c in [
    "periodo", "sexo_str", "grupo_edad",
    "cs_p13_1", "e_con", "tip_con",
] if c in df.columns]

print(f"Columnas numéricas   : {len(COLS_NUMERICAS)}")
print(f"Columnas categóricas : {len(COLS_CATEGORICAS)}")

In [0]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)

resumen = pd.DataFrame({
    'nulos': nulos,
    'porcentaje': nulos_pct
}).sort_values('porcentaje', ascending=False)

resumen[resumen['nulos'] > 0]

In [0]:
# n_hij: nulo significa 0 hijos
df['n_hij'] = df['n_hij'].fillna(0)

# anios_esc y hrsocup: eliminar esas filas
df = df.dropna(subset=['anios_esc', 'hrsocup'])

print(f"Filas después de limpieza: {df.shape[0]:,}")

## 1. Evolución del desempleo juvenil 2020-2025

In [0]:
tasa_desempleo = (
    df.groupby('periodo')
    .agg(
        desempleados=('desempleado', 'sum'),
        pea_total=('pea', 'sum')
    )
    .reset_index()
)
tasa_desempleo['tasa_desempleo'] = (
    tasa_desempleo['desempleados'] / tasa_desempleo['pea_total'] * 100
).round(2)

tasa_desempleo = tasa_desempleo.sort_values('periodo')

matplotlib.use("inline")

plt.figure(figsize=(14, 5))
sns.lineplot(data=tasa_desempleo, x='periodo', y='tasa_desempleo', marker='o', color='steelblue')
plt.xticks(rotation=45)
plt.title('Tasa de desempleo juvenil CDMX 2020-2025')
plt.xlabel('Período')
plt.ylabel('Tasa de desempleo (%)')
plt.tight_layout()
plt.show()

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns

tasa_sexo = (
    df.groupby(['periodo', 'sexo_str'])
    .agg(
        desempleados=('desempleado', 'sum'),
        pea_total=('pea', 'sum')
    )
    .reset_index()
)

tasa_sexo['tasa_desempleo'] = (
    tasa_sexo['desempleados'] / tasa_sexo['pea_total'] * 100
).round(2)

tasa_sexo = tasa_sexo.sort_values('periodo')

plt.figure(figsize=(14, 5))
sns.lineplot(data=tasa_sexo, x='periodo', y='tasa_desempleo',
             hue='sexo_str', marker='o')
plt.xticks(rotation=45)
plt.title('Tasa de desempleo juvenil CDMX por sexo 2020-2025')
plt.xlabel('Período')
plt.ylabel('Tasa de desempleo (%)')
plt.legend(title='Sexo')
plt.tight_layout()
plt.show()

In [0]:
df.head()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

tasa_sexo = (
    df.groupby(['periodo', 'sexo_str'])
    .agg(
        desempleados=('desempleado', 'sum'),
        pea_total=('pea', 'sum')
    )
    .reset_index()
)

tasa_sexo['tasa_desempleo'] = (
    tasa_sexo['desempleados'] / tasa_sexo['pea_total'] * 100
).round(2)

tasa_sexo = tasa_sexo.sort_values('periodo')

plt.figure(figsize=(14, 5))
sns.lineplot(data=tasa_sexo, x='periodo', y='tasa_desempleo',
             hue='sexo_str', marker='o')
plt.xticks(rotation=45)
plt.title('Tasa de desempleo juvenil CDMX por sexo 2020-2025')
plt.xlabel('Período')
plt.ylabel('Tasa de desempleo (%)')
plt.legend(title='Sexo')
plt.tight_layout()
plt.show()

In [0]:
tasa_educacion = (
    df.groupby('anios_esc')
    .agg(
        desempleados=('desempleado', 'sum'),
        pea_total=('pea', 'sum')
    )
    .reset_index()
)
tasa_educacion['tasa_desempleo'] = (
    tasa_educacion['desempleados'] / tasa_educacion['pea_total'] * 100
).round(2)

tasa_educacion = tasa_educacion.sort_values('anios_esc')

plt.figure(figsize=(12, 5))
sns.barplot(data=tasa_educacion, x='anios_esc', y='tasa_desempleo', color='steelblue')
plt.title('Tasa de desempleo por años de escolaridad')
plt.xlabel('Años de escolaridad')
plt.ylabel('Tasa de desempleo (%)')
plt.tight_layout()
plt.show()

In [0]:
df_ocupados = df[(df['pea'] == 1) & (df['ingocup'] > 0)]
p95 = df_ocupados['ingocup'].quantile(0.95)
df_viz = df_ocupados[df_ocupados['ingocup'] <= p95]

plt.figure(figsize=(12, 5))
sns.boxplot(data=df_viz, x='sexo_str', y='ingocup', color='steelblue')
plt.title('Distribución de ingresos por ocupación según sexo (hasta percentil 95)')
plt.xlabel('Sexo')
plt.ylabel('Ingreso mensual (MXN)')
plt.tight_layout()
plt.show()

# Mediana por sexo
print(df_ocupados.groupby('sexo_str')['ingocup'].median())

In [0]:
# Solo ocupados con valor válido de seg_soc
df_ocupados_formal = df[(df['pea'] == 1) & (df['seg_soc'].isin([1, 2]))]

df_ocupados_formal['informal'] = (df_ocupados_formal['seg_soc'] == 2).astype(int)

informalidad = (
    df_ocupados_formal
    .groupby(['periodo', 'sexo_str'])
    .agg(informales=('informal', 'sum'),
         ocupados=('informal', 'count'))
    .reset_index()
)
informalidad['tasa_informalidad'] = (
    informalidad['informales'] / informalidad['ocupados'] * 100
).round(2)

informalidad = informalidad.sort_values('periodo')

plt.figure(figsize=(14, 5))
sns.lineplot(data=informalidad, x='periodo', y='tasa_informalidad',
             hue='sexo_str', marker='o')
plt.xticks(rotation=45)
plt.title('Tasa de informalidad laboral juvenil CDMX por sexo 2020-2025')
plt.xlabel('Período')
plt.ylabel('Tasa de informalidad (%)')
plt.legend(title='Sexo')
plt.tight_layout()
plt.show()

In [0]:
sectores = {
    0: 'No aplica',
    1: 'Agricultura y primario',
    2: 'Industria',
    3: 'Servicios',
    4: 'Gobierno y otros'
}

df_sector = df[(df['pea'] == 1) & (df['ingocup'] > 0) & (df['rama_est1'] > 0)].copy()
df_sector['sector'] = df_sector['rama_est1'].map(sectores)

ingreso_sector = (
    df_sector.groupby('sector')['ingocup']
    .median()
    .sort_values(ascending=False)
    .reset_index()
)
ingreso_sector.columns = ['sector', 'ingreso_mediano']

conteo_sector = (
    df_sector.groupby('sector')['ingocup']
    .count()
    .reset_index()
)
conteo_sector.columns = ['sector', 'jovenes']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=ingreso_sector, x='ingreso_mediano', y='sector',
            ax=axes[0], color='steelblue')
axes[0].set_title('Ingreso mediano por sector')
axes[0].set_xlabel('Ingreso mediano (MXN)')
axes[0].set_ylabel('')

sns.barplot(data=conteo_sector, x='jovenes', y='sector',
            ax=axes[1], color='steelblue')
axes[1].set_title('Jóvenes ocupados por sector')
axes[1].set_xlabel('Número de jóvenes')
axes[1].set_ylabel('')

plt.suptitle('Mercado laboral juvenil CDMX por sector económico')
plt.tight_layout()
plt.show()

In [0]:
subocupacion = (
    df[df['pea'] == 1]
    .groupby(['periodo', 'sexo_str'])
    .agg(subocupados=('sub_o', 'sum'),
         ocupados=('pea', 'sum'))
    .reset_index()
)
subocupacion['tasa_subocupacion'] = (
    subocupacion['subocupados'] / subocupacion['ocupados'] * 100
).round(2)

subocupacion = subocupacion.sort_values('periodo')

plt.figure(figsize=(14, 5))
sns.lineplot(data=subocupacion, x='periodo', y='tasa_subocupacion',
             hue='sexo_str', marker='o')
plt.xticks(rotation=45)
plt.title('Tasa de subocupación juvenil CDMX por sexo 2020-2025')
plt.xlabel('Período')
plt.ylabel('Tasa de subocupación (%)')
plt.legend(title='Sexo')
plt.tight_layout()
plt.show()

## Conclusiones

### 1. Impacto del COVID-19 en el desempleo juvenil
La tasa de desempleo juvenil en CDMX alcanzó su pico máximo en 2021-T1
con 15.4%, resultado del choque económico de la pandemia. La recuperación
fue sostenida desde 2022 hasta alcanzar el mínimo histórico del período
de 6.78% en 2025-T2. Sin embargo la ausencia del dato de 2020-T2,
trimestre del inicio del confinamiento, sugiere que el pico real pudo
haber sido más severo.

### 2. Recuperación desigual por sexo
Los hombres jóvenes recuperaron niveles pre-pandemia de desempleo antes
que las mujeres. En 2025 la tasa masculina llegó a 6% mientras la
femenina se mantuvo alrededor de 8%, una brecha de 2 puntos porcentuales
que persiste a lo largo de todo el período analizado.

### 3. Barreras estructurales por grupo de edad
El grupo de 18-21 años presenta consistentemente la tasa de desempleo
más alta del período, llegando a 22% durante COVID y manteniéndose
entre 9% y 14% en 2025. El grupo de 26-29 años muestra tasas
significativamente menores, lo que indica que las barreras de entrada
al mercado laboral no desaparecen con la recuperación económica general.

### 4. Educación y desempleo sin relación clara
El análisis por años de escolaridad no muestra una relación inversa
consistente entre nivel educativo y desempleo hasta alcanzar niveles
de posgrado (18-19 años de escolaridad). Esto sugiere que el mercado
laboral de CDMX no absorbe eficientemente a los egresados universitarios,
un hallazgo con implicaciones directas para política educativa y laboral.

### 5. Brecha salarial de género
La mediana de ingreso mensual de los hombres jóvenes ocupados es de
$6,880 MXN frente a $6,020 MXN de las mujeres, una brecha de 12.5%.
Esta diferencia persiste controlando por sector económico y se suma
a la mayor tasa de desempleo femenina.

### 6. Informalidad estructural persistente
Más del 50% de los jóvenes ocupados en CDMX trabajan sin seguridad
social en todo el período analizado. Esta tasa no mejoró con la
recuperación post-COVID, lo que confirma que la informalidad es una
característica estructural del mercado laboral juvenil y no un efecto
temporal de la pandemia.

### 7. Concentración en servicios con alta vulnerabilidad
El sector servicios absorbe más del 80% de los jóvenes ocupados en
CDMX, un sector con alta informalidad y volatilidad. Esto explica
en parte las tasas de informalidad y subocupación observadas.

### 8. Triple desventaja femenina
Las mujeres jóvenes en CDMX enfrentan simultáneamente mayor tasa de
desempleo, mayor subocupación cuando trabajan, y menor ingreso mensual.
Esta evidencia, construida con microdatos oficiales del INEGI, apunta
a segregación laboral por género como fenómeno estructural y no coyuntural.

---

### Siguientes pasos
- Notebook 03: Modelo predictivo de desempleo juvenil con CatBoost
- Integración con datos macroeconómicos (INPC, salario mínimo, PIB CDMX)
- Análisis de causalidad entre variables con DoWhy o EconML